# 176. Mixture-of-Depths：怎样给不同 token 动态分配 Transformer 深度？

> **面试问题：MoD 与 MoE 有什么区别？token top-k、固定容量、旁路 residual、训练路由和自回归推理怎样实现？**

## 先给结论

MoE 在同一层选择不同专家，MoD 决定某 token 是否执行这一层的 attention/MLP。固定 top-k capacity 让总计算量可预测，未选 token 沿 residual 旁路；难点在可微路由、因果推理时无法预知未来 token、负载/排序、KV 状态和质量—FLOPs 门禁。

## 推荐回答主线

1. 定义每层 token router、capacity 与稳定 top-k；被选 token 进入重计算块，其他 token 保持 residual。
2. 用 gather→block→scatter 实现 forward，并验证旁路不被误改、梯度能到 router 与 block。
3. 区分训练整序列 top-k 与 decode 在线 threshold/predictor，避免未来 token 决定当前路由。
4. 按 token/layer 切片评估激活率、质量、真实 kernel 利用率、KV 语义和 checkpoint 配置。

## 教学实现边界

教学块只含小 MLP，不复现论文的完整 attention、辅助损失或专用 kernel；单进程 top-k 证明固定预算和状态语义，不代表 gather/scatter 在 GPU 上一定更快。

## 一手资料

- [Mixture-of-Depths](https://arxiv.org/abs/2404.02258)
- [Expert Choice Routing](https://arxiv.org/abs/2202.09368)
- [Learning to Skip for Language Modeling](https://arxiv.org/abs/2311.15436)


In [ ]:
import hashlib
import json
import math
from dataclasses import asdict, dataclass

import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)

import torch
from torch import nn

# 每个 batch 独立选择固定数量 token，故意让长度不能被容量比例整除。
torch.manual_seed(176)
B, L, D, CAPACITY = 2, 7, 10, 3
x = torch.randn(B, L, D)

assert 0 < CAPACITY < L
assert x.shape == (B, L, D)
assert L % CAPACITY != 0


## 1. 稳定 top-k：容量固定，身份动态

router 为每个 token 产生标量分数，每个序列只选择 k 个。相同分数的 tie-break 必须确定，否则重放与分布式 rank 可能不一致；这里用极小位置偏置让较早位置优先。


In [ ]:
def stable_topk(scores, k, valid=None):
    if scores.ndim != 2 or not 0 <= k <= scores.shape[-1]:
        raise ValueError("scores 必须为二维且 k 位于合法范围")
    if valid is None:
        valid = torch.ones_like(scores, dtype=torch.bool)
    if valid.shape != scores.shape:
        raise ValueError("valid 必须与 scores 同形")
    # stable sort 在完全同分时保持原位置顺序，不依赖可能低于 ULP 的 epsilon 扰动。
    masked_scores = scores.masked_fill(~valid, -torch.inf)
    return torch.argsort(masked_scores, dim=-1, descending=True, stable=True)[:, :k]

# 每行恰取 k 个唯一位置；零分和极大数同分时都确定性地让较早位置优先。
ties = torch.zeros(B, L)
selected_ties = stable_topk(ties, CAPACITY)
large_ties = torch.full((1, L), 1e20)
assert selected_ties.shape == (B, CAPACITY)
assert all(len(set(row.tolist())) == CAPACITY for row in selected_ties)
assert selected_ties[0].tolist() == list(range(CAPACITY))
assert stable_topk(large_ties, CAPACITY)[0].tolist() == list(range(CAPACITY))


## 2. 手写 MoD Block：未选 token 精确旁路

被选 token 经过 LayerNorm+MLP，再按 sigmoid router weight 加回 residual；未选 token 原样复制。生产 attention 块还要处理选中 token 的因果位置、KV 和 token 顺序，不能把压缩后的局部序号当原位置。


In [ ]:
class TinyMoDBlock(nn.Module):
    def __init__(self, dim, capacity):
        super().__init__()
        self.capacity = capacity
        self.router = nn.Linear(dim, 1, bias=False)
        self.norm = nn.LayerNorm(dim)
        self.heavy = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))

    def _apply_routes(self, hidden, scores, routed_mask):
        output = hidden.clone()
        for batch_id in range(hidden.shape[0]):
            index = torch.nonzero(routed_mask[batch_id], as_tuple=False).flatten()
            if index.numel() == 0:
                continue
            update = self.heavy(self.norm(hidden[batch_id, index]))
            gate = torch.sigmoid(scores[batch_id, index])[:, None]
            output[batch_id, index] = hidden[batch_id, index] + gate * update
        return output

    def forward(self, hidden, valid_mask=None):
        if valid_mask is None:
            valid_mask = torch.ones(hidden.shape[:2], dtype=torch.bool, device=hidden.device)
        if valid_mask.shape != hidden.shape[:2]:
            raise ValueError("valid_mask 必须与 token 轴同形")
        scores = self.router(hidden).squeeze(-1)
        selected = stable_topk(scores, self.capacity, valid_mask)
        selected_valid = valid_mask.gather(1, selected)
        routed_mask = torch.zeros_like(valid_mask).scatter(1, selected, selected_valid)
        return self._apply_routes(hidden, scores, routed_mask), scores, selected, routed_mask

    def forward_decode(self, hidden, valid_mask, threshold, window, max_active):
        # decode 路由在运行时调用逐 token 在线预算器，不能偷看未来分数。
        scores = self.router(hidden).squeeze(-1)
        routed_mask = online_budget_mask(scores, valid_mask, threshold, window, max_active)
        return self._apply_routes(hidden, scores, routed_mask), scores, routed_mask

# 输出形状不变，选中数固定，至少一个真实路由 token 被更新。
block = TinyMoDBlock(D, CAPACITY)
out, route_scores, selected, routed = block(x)
assert out.shape == x.shape
assert selected.shape == (B, CAPACITY)
assert routed.sum(1).tolist() == [CAPACITY] * B
assert (out[routed] - x[routed]).abs().sum() > 0


## 3. Scatter oracle：所有未选 token 必须逐元素等于输入

动态 gather/scatter 最常见 bug 是 batch 索引错位、排序后没有还原位置或 padding 被选。用布尔 mask 直接验证旁路，可作为融合 kernel 的回归 oracle。


In [ ]:
def selected_mask(indices, length, active=None):
    if active is None:
        active = torch.ones_like(indices, dtype=torch.bool)
    mask = torch.zeros(indices.shape[0], length, dtype=torch.bool, device=indices.device)
    return mask.scatter(1, indices, active)

# gather/scatter 的 routed mask 与模块返回一致；未选位置逐元素完全旁路。
mask = selected_mask(selected, L)
assert torch.equal(mask, routed)
assert torch.equal(out[~routed], x[~routed])
assert routed.sum(1).tolist() == [CAPACITY] * B
assert not (routed & ~mask).any()


## 4. 路由梯度：hard top-k 身份不可导，gate 值仍可导

top-k index 对 router score 的离散变化没有普通梯度，但选中 token 的 sigmoid gate 会给 router 学习信号。论文/实现还可加入辅助 predictor 或 straight-through 技巧；必须明确哪条路径训练路由。


In [ ]:
# 对输出求损失，检查 heavy 与 router 都收到有限梯度。
block.zero_grad(set_to_none=True)
loss = out.square().mean()
loss.backward()
assert block.router.weight.grad is not None and block.router.weight.grad.norm() > 0
assert block.heavy[0].weight.grad is not None and block.heavy[0].weight.grad.norm() > 0
assert torch.isfinite(block.router.weight.grad).all()


## 5. Padding 与 capacity：只在有效 token 中竞争

短样本 padding 不应占计算名额。每个样本有效长度不同时，固定全局 k 需取 `min(k, valid_count)` 并用 padded gather 或分桶；否则会浪费算力或产生全无效 top-k。


In [ ]:
# 让 padding 得分远高于有效 token，真实 block 仍只能路由有效位置且数目等于 min(valid_count, capacity)。
valid = torch.tensor([[1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0]], dtype=torch.bool)
padding_probe = x.clone()
router_direction = block.router.weight.detach()[0]
padding_probe[~valid] = router_direction * 1e6
masked_out, masked_scores, masked_selected, masked_routed = block(padding_probe, valid)
expected_count = valid.sum(1).clamp_max(CAPACITY)
assert masked_scores[~valid].min() > masked_scores[valid].max()
assert torch.equal(masked_routed.sum(1), expected_count)
assert not masked_routed[~valid].any()
assert torch.equal(masked_out[~valid], padding_probe[~valid])
assert all(valid[b, masked_selected[b][valid[b, masked_selected[b]]]].all() for b in range(B))


## 6. 训练 top-k 与自回归 decode：未来 token 不得改变当前决策

整序列 top-k 会让位置 t 是否入选依赖未来 token 的分数，不可直接用于在线 decode。可训练局部 threshold/predictor，在每个新 token 到来时独立决定，并用滑动预算控制长期激活率。


In [ ]:
class OnlineBudgetRouter:
    def __init__(self, threshold, window, max_active):
        if window <= 0 or not 0 <= max_active <= window:
            raise ValueError("window/max_active 预算非法")
        self.threshold, self.window, self.max_active = threshold, window, max_active
        self.history = []

    def decide(self, score, valid=True):
        recent = self.history[-(self.window - 1):] if self.window > 1 else []
        allowed = sum(recent) < self.max_active
        active = bool(valid and score >= self.threshold and allowed)
        self.history.append(active)
        return active

def online_budget_mask(scores, valid, threshold, window, max_active):
    if scores.shape != valid.shape:
        raise ValueError("decode scores/valid 必须同形")
    result = torch.zeros_like(valid)
    for batch_id in range(scores.shape[0]):
        router = OnlineBudgetRouter(threshold, window, max_active)
        for position in range(scores.shape[1]):
            result[batch_id, position] = router.decide(float(scores[batch_id, position].detach()), bool(valid[batch_id, position]))
    return result

# 真实 decode forward 消费在线决策：padding 永不激活、窗口预算成立、未来变化不影响过去路由。
decode_out, decode_scores, decode_routed = block.forward_decode(x, valid, threshold=-10.0, window=4, max_active=2)
assert not decode_routed[~valid].any()
assert torch.equal(decode_out[~decode_routed], x[~decode_routed])
assert all(decode_routed[:, start:start + 4].sum(1).max() <= 2 for start in range(L - 3))

future_changed = x.clone(); future_changed[:, 4:] += 1000
_, _, changed_routed = block.forward_decode(future_changed, valid, threshold=-10.0, window=4, max_active=2)
assert torch.equal(decode_routed[:, :4], changed_routed[:, :4])
assert decode_scores.shape == (B, L)


## 7. FLOPs 与 wall-clock：省算术不保证省时间

若 dense block 每 token 成本 C，理想 heavy FLOPs 从 `L*C` 降到 `k*C`，但 router、gather/scatter、padding、kernel 启动与小矩阵效率会吞掉收益。应以端到端 tokens/s、TPOT 和能耗验收。


In [ ]:
def ideal_compute_ratio(length, capacity, router_cost_ratio=0.02):
    return capacity / length + router_cost_ratio

def realized_speedup(dense_ms, router_ms, gather_ms, sparse_heavy_ms):
    sparse_ms = router_ms + gather_ms + sparse_heavy_ms
    return dense_ms / sparse_ms

# 理想比率随 capacity 上升；有开销时实际 speedup 可能小于 1。
ratio = ideal_compute_ratio(L, CAPACITY)
assert 0 < ratio < 1
assert ideal_compute_ratio(L, CAPACITY + 1) > ratio
assert realized_speedup(1.0, 0.2, 0.4, 0.6) < 1.0


## 8. 发布门禁：router recipe 与层位置属于模型结构

capacity、是否共享 router、训练 top-k 与 decode predictor 的映射、原始位置编码和 KV 处理都要进 manifest。质量评测按 token 类型、语言、长上下文和推理任务切片，并对 dense baseline 做同预算比较。


In [ ]:
@dataclass(frozen=True)
class MoDConfig:
    model_dim: int
    sequence_capacity: int
    router: str
    decode_policy: str
    mod_layers: tuple[int, ...]

def artifact_digest(config):
    return hashlib.sha256(json.dumps(asdict(config), sort_keys=True).encode()).hexdigest()

# 改容量或 decode 策略必须产生新结构摘要，配置容量与模块一致。
config = MoDConfig(D, CAPACITY, "stable-topk-padding-v2", "causal-window-budget-v2", (1, 3, 5))
digest = artifact_digest(config)
assert config.sequence_capacity == block.capacity
assert len(digest) == 64
assert digest != artifact_digest(MoDConfig(D, CAPACITY + 1, config.router, config.decode_policy, (1, 3, 5)))


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
